# Team Tokens: Project 1 runs on Google Colab

This notebook runs the GPU parts of the project and then refreshes the analyses and figures.

**Before you start:** Runtime > Change runtime type > **T4 GPU**.

| Step | What it does | Time on a T4 |
|---|---|---|
| 1 | Clone the repository and install the right packages | 2 min |
| 2 | Quick checks (nothing is saved) | 3 min |
| 3 | DeBERTa-v3-small again, now also exporting test predictions and saving the model for the demo | ~27 min |
| 4 | TextCNN and BiLSTM with GloVe (Week 3 methods) | ~10 min |
| 5 | Error analysis and figures (CPU) | 1 min |
| 6 | Live demo | - |
| 7 | Download the new result files | - |

Re-running DeBERTa on a GPU can change its scores in the last decimals (GPU kernels are not fully deterministic). Commit the new `transformer_benchmark.csv`, `transformer_metrics.json` and `transformer_predictions_*.csv` together, so the numbers and the predictions always come from the same run, and update the README table if the numbers changed.

## 1. Setup

In [ ]:
!git clone https://github.com/zehraoz36/tokens-project-1.git
%cd tokens-project-1
# Colab's preinstalled torchvision does not match and breaks the transformers import (we do not use it)
!pip uninstall -y -q torchvision
!pip install -q transformers==5.18.0 nltk
!python -m nltk.downloader -q wordnet omw-1.4
import torch; print(torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch the runtime to T4 GPU")

If you changed files locally that are not pushed yet, upload them now (Files panel on the left), or push first and re-run the cell above.

## 2. Quick checks (about 3 minutes, saves nothing)

In [ ]:
!python train_transformer.py --limit 200 --epochs 1
!python train_neural.py --model cnn --limit 300 --epochs 1
!python train_neural.py --model bilstm --limit 300 --epochs 1

## 3. DeBERTa-v3-small: full run with prediction export and saved model

In [ ]:
!python train_transformer.py --save-model models/deberta
!cat results/transformer_benchmark.csv

## 4. Week 3 neural baselines: TextCNN and BiLSTM with GloVe 100d

In [ ]:
!python train_neural.py --model cnn
!python train_neural.py --model bilstm
!cat results/neural_cnn_benchmark.csv results/neural_bilstm_benchmark.csv

## 5. Error analysis and figures

`experiments.py` (classical feature engineering, about 15 min) and `paper_protocol.py` (benchmark against the source paper, about 25 min) run on CPU; their results are already in the repository, so re-running them is optional.

In [ ]:
# !python experiments.py       # optional, ~15 min
# !python paper_protocol.py    # optional, ~25 min
!python error_analysis.py
!python make_figures.py
from IPython.display import Image, display
for name in ["model_comparison", "confusion_matrices", "error_cues", "paper_comparison", "class_distribution"]:
    display(Image(f"results/figures/{name}.png", width=900))

## 6. Live demo

In [ ]:
examples = [
    "TypeError when calling df.groupby with an empty list",
    "Add support for zstd compression in to_parquet",
    "Segfault in scipy.linalg.svd for very large matrices",
]
for title in examples:
    !python predict.py --model models/deberta "{title}"
# the classical model also shows which words pushed the decision:
!python predict.py "TypeError when calling df.groupby with an empty list"

In [ ]:
# Try your own issue title:
title = "Plot legend overlaps the axis labels when using tight_layout"
!python predict.py --model models/deberta "{title}"

## 7. Download the new result files

In [ ]:
!zip -q -r new_results.zip results
from google.colab import files
files.download("new_results.zip")

Unzip `new_results.zip` into your local clone of the repository (it replaces the `results/` folder), check the changes in GitHub Desktop, then commit and push. The saved model in `models/` is not committed (it is about 550 MB and is listed in `.gitignore`).